# 14 - Evaluacion final en test real

Este notebook valida los modelos finales sobre `data/processed/test_processed.csv`, manteniendo la distribucion real de `Revenue` cercana al 15%.

A diferencia de la version anterior, aqui no se asume un unico modelo final desde el inicio. Se comparan Random Forest, XGBoost y LightGBM en el mismo test, y se define una recomendacion segun escenario de negocio.


In [ ]:
import os
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "test_processed.csv"
MODELS_DIR = ROOT / "models"
RESULTS_PATH = MODELS_DIR / "final_test_model_comparison.csv"
SEGMENT_RESULTS_PATH = MODELS_DIR / "final_test_by_visitor_type.csv"

print("ROOT:", ROOT)
print("TEST:", DATA_PATH)


## Carga de test y modelos


In [ ]:
TARGET_COL = "Revenue"

df_test = pd.read_csv(DATA_PATH)
if TARGET_COL not in df_test.columns:
    raise ValueError(f"No se encontro la columna target '{TARGET_COL}'.")

X_test = df_test.drop(columns=[TARGET_COL])
y_test = df_test[TARGET_COL].astype(int)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("\nDistribucion real en test:")
print(y_test.value_counts().sort_index())
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
MODEL_PATHS = {
    ("Random Forest", "final_test"): MODELS_DIR / "best_tuned_model.pkl",
    ("XGBoost", "base"): MODELS_DIR / "xgboost_base.pkl",
    ("XGBoost", "tuned"): MODELS_DIR / "tuned_xgboost.pkl",
    ("LightGBM", "base"): MODELS_DIR / "lightgbm_base.pkl",
    ("LightGBM", "tuned"): MODELS_DIR / "tuned_lightgbm.pkl",
}

models = {}
missing_models = []

for key, path in MODEL_PATHS.items():
    if path.exists():
        models[key] = joblib.load(path)
    else:
        missing_models.append(str(path))

print("Modelos cargados:")
for model_name, version in models:
    print(f"- {model_name} ({version})")

if missing_models:
    print("\nModelos no encontrados:")
    for path in missing_models:
        print("-", path)


def assert_feature_compatibility(model, X, label):
    if hasattr(model, "feature_names_in_"):
        expected = list(model.feature_names_in_)
        current = list(X.columns)
        if expected != current:
            raise ValueError(
                f"{label} no es compatible con X_test. "
                f"Esperaba {len(expected)} columnas y recibio {len(current)}."
            )

for key, model in models.items():
    assert_feature_compatibility(model, X_test, f"{key[0]} {key[1]}")

print("\nTodos los modelos cargados son compatibles con X_test.")


## Evaluacion general en test real

Se evalua cada modelo con `threshold=0.50` y `threshold=0.25` para comparar una politica conservadora contra una politica orientada a mayor captura de compradores.


In [ ]:
THRESHOLDS = [0.50, 0.25]


def evaluate_model_test(model, X, y, threshold):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred),
        "f1": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_proba),
        "pr_auc": average_precision_score(y, y_proba),
        "predicted_positive": int(y_pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


general_rows = []

for (model_name, version), model in models.items():
    for threshold in THRESHOLDS:
        general_rows.append({
            "model": model_name,
            "version": version,
            **evaluate_model_test(model, X_test, y_test, threshold),
        })

general_results_df = pd.DataFrame(general_rows).sort_values(
    ["threshold", "f1"],
    ascending=[False, False],
).reset_index(drop=True)

general_results_df.to_csv(RESULTS_PATH, index=False)
print("Resultados generales guardados en:", RESULTS_PATH)
display(general_results_df)


In [ ]:
comparison_cols = [
    "model",
    "version",
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]

display(general_results_df[comparison_cols])


## Matriz de confusion y reporte del mejor modelo global conservador

Para una politica global conservadora se toma el mejor modelo con `threshold=0.50`, priorizando PR-AUC, ROC-AUC y precision. Este criterio favorece modelos buenos para ranking y campanas con costo de contacto.


In [ ]:
conservative_candidates = general_results_df[general_results_df["threshold"] == 0.50].copy()
conservative_best = conservative_candidates.sort_values(
    ["pr_auc", "roc_auc", "precision", "f1"],
    ascending=False,
).iloc[0]

best_key = (conservative_best["model"], conservative_best["version"])
best_model = models[best_key]
best_threshold = conservative_best["threshold"]
y_proba_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_proba_best >= best_threshold).astype(int)

print("Modelo global conservador recomendado:")
print(f"- Modelo: {best_key[0]} ({best_key[1]})")
print(f"- Threshold: {best_threshold}")
print(f"- PR-AUC: {conservative_best['pr_auc']:.4f}")
print(f"- ROC-AUC: {conservative_best['roc_auc']:.4f}")
print(f"- Precision: {conservative_best['precision']:.4f}")
print(f"- Recall: {conservative_best['recall']:.4f}")
print(f"- F1: {conservative_best['f1']:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred_best, target_names=["No Revenue", "Revenue"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best,
    display_labels=["No Revenue", "Revenue"],
    cmap="Blues",
)
plt.title(f"Matriz de confusion - {best_key[0]} {best_key[1]} threshold={best_threshold}")
plt.show()


## Evaluacion segmentada por VisitorType

Se separa el test por tipo de visitante. La decision de negocio puede cambiar porque contactar usuarios nuevos suele ser mas caro que activar usuarios recurrentes.


In [ ]:
VISITOR_TYPE_COLS = [col for col in X_test.columns if col.startswith("cat__VisitorType_")]

if not VISITOR_TYPE_COLS:
    raise ValueError("No se encontraron columnas one-hot de VisitorType en X_test.")

print("Distribucion por VisitorType en test:")
print(X_test[VISITOR_TYPE_COLS].sum().sort_values(ascending=False))


def safe_segment_metrics(y_true, y_pred, y_proba):
    positives = int(y_true.sum())
    negatives = int((y_true == 0).sum())

    metrics = {
        "segment_size": int(len(y_true)),
        "segment_buyers": positives,
        "segment_non_buyers": negatives,
        "segment_conversion_rate": float(y_true.mean()) if len(y_true) else np.nan,
        "accuracy": accuracy_score(y_true, y_pred) if len(y_true) else np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan,
        "recall": recall_score(y_true, y_pred, zero_division=0) if positives > 0 else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan,
        "predicted_positive": int(y_pred.sum()),
        "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
        "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
        "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
    }

    if positives > 0 and negatives > 0:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        metrics["pr_auc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan

    return metrics


In [ ]:
segment_rows = []

for (model_name, version), model in models.items():
    y_proba_full = model.predict_proba(X_test)[:, 1]

    for col in VISITOR_TYPE_COLS:
        visitor_type = col.replace("cat__VisitorType_", "")
        mask = X_test[col] == 1

        if mask.sum() == 0:
            continue

        y_segment = y_test.loc[mask].astype(int)
        proba_segment = y_proba_full[mask.to_numpy()]

        for threshold in THRESHOLDS:
            pred_segment = (proba_segment >= threshold).astype(int)
            segment_rows.append({
                "visitor_type": visitor_type,
                "model": model_name,
                "version": version,
                "threshold": threshold,
                **safe_segment_metrics(y_segment, pred_segment, proba_segment),
            })

segment_results_df = pd.DataFrame(segment_rows).sort_values(
    ["visitor_type", "threshold", "f1"],
    ascending=[True, False, False],
).reset_index(drop=True)

segment_results_df.to_csv(SEGMENT_RESULTS_PATH, index=False)
print("Resultados segmentados guardados en:", SEGMENT_RESULTS_PATH)
display(segment_results_df)


## Recomendacion por escenario

La seleccion no se hace solo por F1 global. Se define una politica segun el costo de contacto y la dificultad de activar cada segmento.


In [ ]:
def pick_recommendations(general_df, segment_df):
    recommendations = []

    global_conservative = general_df[general_df["threshold"] == 0.50].sort_values(
        ["pr_auc", "roc_auc", "precision", "f1"],
        ascending=False,
    ).iloc[0]
    recommendations.append({
        "scenario": "Global conservador / ranking de compradores",
        "selection_reason": "Mayor PR-AUC y ROC-AUC en test real; util cuando el contacto tiene costo",
        **global_conservative.to_dict(),
    })

    max_capture = general_df[general_df["threshold"] == 0.25].sort_values(
        ["recall", "f1"],
        ascending=False,
    ).iloc[0]
    recommendations.append({
        "scenario": "Maxima captura de compradores",
        "selection_reason": "Mayor recall; util si perder compradores es mas caro que contactar falsos positivos",
        **max_capture.to_dict(),
    })

    new_visitors = segment_df[
        (segment_df["visitor_type"] == "New_Visitor") &
        (segment_df["threshold"] == 0.50)
    ].sort_values(["precision", "pr_auc", "f1"], ascending=False)
    if not new_visitors.empty:
        selected = new_visitors.iloc[0]
        recommendations.append({
            "scenario": "Usuarios nuevos",
            "selection_reason": "Prioriza precision porque contactar nuevos suele ser mas dificil y caro",
            **selected.to_dict(),
        })

    returning_visitors = segment_df[
        (segment_df["visitor_type"] == "Returning_Visitor") &
        (segment_df["threshold"] == 0.25)
    ].sort_values(["recall", "f1", "precision"], ascending=False)
    if not returning_visitors.empty:
        selected = returning_visitors.iloc[0]
        recommendations.append({
            "scenario": "Usuarios recurrentes",
            "selection_reason": "Prioriza recall porque la activacion suele ser mas barata",
            **selected.to_dict(),
        })

    return pd.DataFrame(recommendations)


recommendations_df = pick_recommendations(general_results_df, segment_results_df)

recommendation_cols = [
    "scenario",
    "selection_reason",
    "visitor_type",
    "model",
    "version",
    "threshold",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]

existing_cols = [col for col in recommendation_cols if col in recommendations_df.columns]
display(recommendations_df[existing_cols])


## Criterios de exito y cierre

Los criterios tecnicos iniciales eran exigentes para un problema con solo ~15% de compras. En test real, ningun modelo cumple simultaneamente precision > 0.80, recall > 0.80 y F1 > 0.85 a nivel global. Sin embargo, la evaluacion segmentada si produce politicas accionables:

- Para `New_Visitor`, usar XGBoost con politica conservadora permite alta precision y pocos falsos positivos.
- Para `Returning_Visitor`, usar Random Forest con threshold bajo aumenta la captura de compradores, aceptando mas falsos positivos.
- Para una campana global con costo de contacto, XGBoost tuneado es mejor candidato por PR-AUC y capacidad de ranking.
- Para maxima captura global, Random Forest con threshold bajo sigue siendo util.

La recomendacion final no es un unico modelo universal, sino una estrategia de decision por escenario de negocio.


In [ ]:
print("Resumen ejecutivo final")
print("- Test real:", X_test.shape[0], "sesiones")
print("- Compradores reales:", int(y_test.sum()), f"({y_test.mean():.2%})")
print("- Modelo global conservador:", f"{best_key[0]} ({best_key[1]})", "threshold", best_threshold)

for _, row in recommendations_df.iterrows():
    print("\nEscenario:", row["scenario"])
    print("Modelo:", f"{row['model']} ({row['version']})")
    print("Threshold:", row["threshold"])
    print("Precision:", round(row["precision"], 4) if pd.notna(row["precision"]) else "NA")
    print("Recall:", round(row["recall"], 4) if pd.notna(row["recall"]) else "NA")
    print("F1:", round(row["f1"], 4) if pd.notna(row["f1"]) else "NA")
    print("Motivo:", row["selection_reason"])
